# 1. Your first SCRAPS run

The shortest path from "I have an ASE primitive cell" to "I have a special
quasi-random structure (SQS) supercell" using the `scraps_ase` wrapper.

We'll build a 108-atom FCC NiCoCr SQS — the classic equiatomic CCA test case.


## Setup

`tools/scraps_ase.py` ships with the repo. We add `tools/` to the path so
imports work without an install step.


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent  # this notebook lives in notebooks/
sys.path.insert(0, str(REPO / "tools"))

from ase.build import bulk
from scraps_ase import run_sqs_from_atoms


## Build the primitive cell

ASE's `bulk` helper gives us a 4-atom conventional FCC cell. SCRAPS will tile
this by `Cell_DIM` to form the supercell.


In [ ]:
primitive = bulk("Cu", "fcc", a=3.56, cubic=True)
print(primitive)
print("Primitive atoms:", len(primitive))


## Run SCRAPS

`run_sqs_from_atoms` writes the POSCAR + INPUT, finds the right `mpirun`,
launches scraps, and reads back `SCRAPS.vasp` as an ASE Atoms object with
proper species labels.

`element_counts=[36, 36, 36]` ⇒ 108 atoms total (= 4 basis × 3³ cells).
`max_shellnum=2` weights the first two coordination shells equally. `rng_seed`
makes the run reproducible across machines.


In [ ]:
sqs = run_sqs_from_atoms(
    primitive,
    species=["Ni", "Co", "Cr"],
    element_counts=[36, 36, 36],
    cell_dim=(3, 3, 3),
    max_shellnum=2,
    weights=[1, 1],
    rng_seed=42,
    work_dir=Path.cwd() / "nb1_first_run",
)
print("Atoms:", len(sqs))
print("Symbols:", sqs.symbols)
print("Final fitness:", sqs.info.get("scraps_final_fitness"))


## Verify with the independent SRO calculator

`verify_sro.py` reads `SCRAPS.vasp` and computes the Warren-Cowley α matrix
directly via pymatgen — no shared code with the C++ optimizer, so it's a true
cross-check. For an SQS we want α ≈ 0 at every entry of every weighted shell.


In [ ]:
import subprocess
verify = REPO / "tools" / "verify_sro.py"
work_dir = sqs.info["scraps_work_dir"]
result = subprocess.run(
    [sys.executable, str(verify),
     str(Path(work_dir) / "SCRAPS.vasp"),
     "--species", "Ni,Co,Cr", "--shells", "2"],
    capture_output=True, text=True,
)
print(result.stdout)


## Use the output

`sqs` is a standard ASE Atoms object — save in any format ASE supports.


In [ ]:
out = Path.cwd() / "ni_co_cr_sqs.cif"
sqs.write(str(out))
print(f"Wrote {out}")
print(f"Lattice: {sqs.cell.cellpar()}")


That's it — a 108-atom NiCoCr SQS in a few lines. The next notebook covers
targeting **non-zero** SRO values (literature CSRO patterns); the third covers
multi-sublattice HEB structures.
